# RuleTreeRank local explanation

This notebook trains a small RuleTreeRank example on FindHR and inspects one prediction with `explain_local`. The last section regenerates the static PDF produced by `make_ruletreerank_explanation.py`, so the raw explanation dictionary can be read together with a compact visual summary.


Load autoreload and make the repository packages importable from the notebook directory.


In [ ]:
%reload_ext autoreload
%autoreload 2

import sys

sys.path.append("../")

Import the dataset helpers, model-parameter wrapper, RuleTree base estimator, and RuleTreeRank components used in the example.


In [ ]:
from pathlib import Path
from ltr_utility import ModelParam
from ltr_utility.dataset import load_by_query_dataset, DatasetName

from RuleTree import RuleTreeRegressor
from ruletreerank import RuleTreeRank, PairwiseDistanceTree, KNNRegFast

Load a small FindHR subset. Keeping only a few queries makes the explanation workflow fast while preserving the same `x`, `y`, `q`, and feature-name interface used by the full experiments.


In [ ]:
base_path = Path("../datasets")
dataset_df = load_by_query_dataset(base_path, DatasetName.FINDHR, get_first=3)

Configure and fit RuleTreeRank. The shallow `RuleTreeRegressor` gives the initial interpretable score and leaf assignment; `PairwiseDistanceTree` learns the pairwise distance used by the KNN residual-correction stage; `KNNRegFast` aggregates the nearest-neighbor corrections.


In [ ]:
rtr = RuleTreeRank(
    distance_f=ModelParam(PairwiseDistanceTree, {
        "base_regressor": ModelParam(RuleTreeRegressor, {"max_depth": 3}),
        "feature_concat": True,
        "feature_diff": False,
        "feature_sq_diff": False,
        "subsample": 1.,
        "verbose": False,
    }),
    aggregation_f=ModelParam(KNNRegFast, {"n_neighbors": 3, "n_jobs": 1}),
    base_regressor=RuleTreeRegressor(max_depth=3, max_leaf_nodes=4,
                                     min_samples_split=4),
    dist_objective="dist",
    verbose=False,
    n_jobs_leaf=10,
).fit(dataset_df.x, dataset_df.y, dataset_df.q)

Pick one row to explain. `explain_local` expects a single instance, so the double-bracket indexing keeps the result as a 2D matrix.


In [ ]:
example = dataset_df.x[[1]]
example

Display the feature names that will be used to replace raw `X_i` references in the human-readable rules.


In [ ]:
dataset_df.feature_names()

Generate the local explanation dictionary. It contains the shallow-tree score and path, the nearest neighbors used for the correction, their scores, the learned pairwise distances, and the matched PDT rules.


In [ ]:
p = rtr.explain_local(example, columns_name=dataset_df.feature_names())
p

## Graphical summary

`make_ruletreerank_explanation.py` renders a static PDF schematic of the same explanation concepts: similar instances, the neighborhood graph, matched rules, residual correction, and final score. The figure is illustrative and should be regenerated when the visual design changes.


In [ ]:
from IPython.display import IFrame, display

import make_ruletreerank_explanation as explanation_figure

explanation_figure.main()
display(IFrame("../imgs/explanation.pdf", width=820, height=520))
